In [ ]:
import pandas as pd
from openai import OpenAI
import time
import os
from dotenv import load_dotenv

load_dotenv()

In [2]:
df = pd.read_csv('transactions.csv', skiprows=8)
print(df)

  Reference Code Date Time                                     Description  \
0        14R9U33   50:04.0            Paid for Daraz Kaymu Private Limited   
1        144MMDE   49:26.0               Charge on payment for Electricity   
2        144MMDE   49:26.0                           Paid For NEA - 101782   
3        144MLNL   49:04.0               Charge on payment for Electricity   
4        144MLNL   49:03.0                           Paid For NEA - 101783   
5        144MKEF   48:23.0          Money transferred from NABIL BANK LTD.   
6        13XV69Q   15:48.0                           Bank transfer charges   
7        13XV69Q   15:48.0  Money transferred to MACHHAPUCHCHHRE BANK LTD.   
8        13WIRP7   36:08.0              Topup for NTC Namaste - 9861465063   
9        12NR95N   32:44.0         Money transferred from KUMARI BANK LTD.   

       Dr.   Cr.    Status  Balance (NPR)               Channel  
0   676.00     0  COMPLETE           3.25  Linked Esewa Payment  
1     5.6

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Sample dataframe
data = {
    "Description": [
        "Paid to big movies",
        "paid to supercell for extra gems",
        "paid fees to st xaviers college",
        "paid for pathao",
        "paid for vegetables in bigmart",
        "topup charge to NTC"
    ]
}

df = pd.DataFrame(data)

# Function to classify a single transaction
def classify_transaction(description):
    prompt = f"""
You are a financial transaction classifier.
Classify the following transaction description into one of these categories:
Groceries & Shopping, Banking & Finance, Dining & Food, Income,
Subscriptions, Personal Care, Entertainment, Travel, Education, others.

Transaction description: "{description}"

Only reply with the category name.
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        category = response.choices[0].message.content.strip()
        return category
    except Exception as e:
        print(f"Error classifying '{description}': {e}")
        return "Error"

# Apply classification to the dataframe
categories = []
for desc in df['Description']:
    category = classify_transaction(desc)
    categories.append(category)
    time.sleep(1)  # small delay to avoid rate limits

df['Predicted_Category'] = categories

print(df)

                        Description    Predicted_Category
0                Paid to big movies         Entertainment
1  paid to supercell for extra gems         Entertainment
2   paid fees to st xaviers college             Education
3                   paid for pathao                Travel
4    paid for vegetables in bigmart  Groceries & Shopping
5               topup charge to NTC     Banking & Finance
